In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import driver_genes as dg
dg.set_verbosity(False)
from driver_genes.pipelines import Pipeline

In [2]:
pp = Pipeline(config_path='config_tutorial.yaml')
pp.fit()
pp.save_best_model()
metrics_df, adata_pred = pp.evaluate(num_results=5, concat=True, agg=True)
metrics_df.to_csv(os.path.join(pp.args.trainer.output_dir, 'test_epoch_metrics.csv'), index=False)
os.makedirs(os.path.join(pp.args.trainer.output_dir, 'anndata'), exist_ok=True)
for key in adata_pred.keys():
    adata_pred[key].write(os.path.join(
        pp.args.trainer.output_dir, 
        'anndata', 
        f'{key}.h5ad'
    ))

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name               | Type                 | Params | Mode  | FLOPs
----------------------------------------------------------------------------
0 | model              | DriverGeneFinder     | 5.9 M  | train | 0    
1 | train_metrics_func | MetricCollection     | 0      | train | 0    
2 | val_metrics_func   | MetricCollection     | 0      | train | 0    
3 | test_metrics_func  | MetricCollection     | 0      | train | 0    
4 | mixing_losses      | UncertaintyWeighting | 3      | train | 0    
----------------------------------------------------------------------------
5.9 M     Trainable params
0         Non-trainable params
5.9 M    

Epoch 40: 100%|██████████| 7/7 [00:00<00:00,  8.61it/s, v_num=1, RelPos/Tr/a=1.000, label#AUROC/Tr/a=1.000, RelPos/Va/a=0.987, label#AUROC/Va/a=0.995]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: 100%|██████████| 7/7 [00:07<00:00,  0.94it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Testing DataLoader 0: 100%|██████████| 7/7 [00:08<00:00,  0.86it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: 100%|██████████| 7/7 [00:07<00:00,  0.94it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: 100%|██████████| 7/7 [00:07<00:00,  0.94it/s]


Show the main metrics for the OOD prediction results.

In [9]:
(
    metrics_df
    .query('Group == "OOD"')
    [[
        'prefix', 'seed',
        'label#AUROC', 'label#AUPRC', 'label#F1',
        'RelPos', 'ACC@Top1', "sample#Recall@Top5"
    ]]
)

,prefix,seed,label#AUROC,label#AUPRC,label#F1,RelPos,ACC@Top1,sample#Recall@Top5
1,preprocessed_Resting,0,0.993954,0.945571,0.842749,0.976793,0.864004,0.930973
4,preprocessed_Resting,1,0.993219,0.936611,0.834846,0.976680,0.857025,0.928664
7,preprocessed_Resting,2,0.992605,0.941157,0.837361,0.976337,0.867273,0.933910
10,preprocessed_Resting,3,0.993164,0.940773,0.838599,0.975057,0.865249,0.927248
13,preprocessed_Resting,4,0.993522,0.945331,0.843713,0.977721,0.869588,0.932483


Compute the prediction for each adata in adata_pred

In [10]:
adata_pred

{'preprocessed_Re-stimulated': AnnData object with n_obs × n_vars = 1107 × 5000
     obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'condition', 'guide_id', 'gene', 'gene_category', 'crispr', 'donor', 'percent.mt', 'percent.ribo', 'nCount_SCT', 'nFeature_SCT', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'CD4.CD8.Score', 'CD4.or.CD8', 'seurat_clusters', 'cluster_name', 'perturbation', 'split', 'bc'
     var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
     uns: 'hvg', 'organism', 'pertgenes'
     obsm: 'latent', 'shift', 'proba'
     layers: 'X_hat', 'dy',
 'preprocessed_Resting': AnnData object with n_obs × n_vars = 3116 × 5000
     obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'condition', 'guide_id', 'gene', 'gene_category', 'crispr', 'donor', 'percent.mt', 'percent.ribo', 'nCount_SCT', 'nFeature_SCT', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'CD4.CD8.Score', 'CD4.or.CD8', 'seurat_clusters', 'cluster_name', 'perturbation', 'split', 'b

In [12]:
from driver_genes.data.utils import compute_pred

In [13]:
for key, ad in adata_pred.items():
    compute_pred(ad)
adata_pred

{'preprocessed_Re-stimulated': AnnData object with n_obs × n_vars = 1107 × 5000
     obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'condition', 'guide_id', 'gene', 'gene_category', 'crispr', 'donor', 'percent.mt', 'percent.ribo', 'nCount_SCT', 'nFeature_SCT', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'CD4.CD8.Score', 'CD4.or.CD8', 'seurat_clusters', 'cluster_name', 'perturbation', 'split', 'bc', 'pred_perturbation'
     var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
     uns: 'hvg', 'organism', 'pertgenes'
     obsm: 'latent', 'shift', 'proba', 'rank', 'RelPos', 'logit'
     layers: 'X_hat', 'dy',
 'preprocessed_Resting': AnnData object with n_obs × n_vars = 3116 × 5000
     obs: 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'condition', 'guide_id', 'gene', 'gene_category', 'crispr', 'donor', 'percent.mt', 'percent.ribo', 'nCount_SCT', 'nFeature_SCT', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'CD4.CD8.Score', 'CD4.or.CD8', 'seurat_cluste